# Visualize Task05 Results
Load `data/experiment/cyl_exp_recon.npz` and visualize vn / vs ground truth and reconstruction for the **cylinder experimental** case.

In [7]:
import numpy as np
from sf_recon.utils.io import load_npz
print('Ready to load Task05 results')

Ready to load Task05 results


In [8]:
import numpy as np

result_path = '../data/experiment/cyl_exp_recon.npz'
with np.load(result_path, allow_pickle=True) as data:
    print('=== NPZ keys ===')
    print(sorted(data.files))

    for key in ['gt', 'recon', 'loss_mask', 'reset_mask',
                'v_gt_speed', 'v_recon_speed', 'RAD',
                'cyl_center_x', 'cyl_center_y']:
        value = data.get(key)
        if value is None:
            print(f'{key}: None')
        else:
            arr = np.asarray(value)
            print(f'{key}: shape={arr.shape}, dtype={arr.dtype}, val={arr}' if arr.ndim == 0 else
                  f'{key}: shape={arr.shape}, dtype={arr.dtype}')

    gt = data.get('gt')
    recon = data.get('recon')
    loss_mask = data.get('loss_mask')

def _safe_array(x):
    if x is None:
        return None
    try:
        return np.asarray(x)
    except Exception:
        return None

gt_arr = _safe_array(gt)
recon_arr = _safe_array(recon)
mask_arr = _safe_array(loss_mask)

print('\n=== Marker preview ===')
for name, arr in [('gt', gt_arr), ('recon', recon_arr), ('loss_mask', mask_arr)]:
    if arr is None:
        print(f'{name}: unavailable')
    else:
        print(f'{name}: ndim={arr.ndim}, shape={arr.shape}')

def _visible_count(points, mask, frame_idx):
    if points is None or points.ndim != 3 or frame_idx >= points.shape[0]:
        return None
    p = np.asarray(points[frame_idx])
    if p.ndim != 2 or p.shape[-1] != 2:
        return None
    if mask is None or mask.ndim < 2 or frame_idx >= mask.shape[0]:
        return len(p)
    m = np.asarray(mask[frame_idx]).reshape(-1) > 0.5
    if len(m) != len(p):
        return f'mask-mismatch points={len(p)} mask={len(m)}'
    return int(np.sum(m))

for i in range(min(5, gt_arr.shape[0] if gt_arr is not None else 0)):
    gt_count = _visible_count(gt_arr, mask_arr, i)
    recon_count = _visible_count(recon_arr, mask_arr, i)
    print(f'frame {i}: gt_visible={gt_count}, recon_visible={recon_count}')

=== NPZ keys ===
['Lx', 'Ly', 'RAD', 'Wx', 'Wy', 'cyl_center_x', 'cyl_center_y', 'fit_continuous_field', 'fit_times', 'gt', 'gt_vel', 'loss_mask', 'marker_x_max', 'marker_x_min', 'marker_y_max', 'marker_y_min', 'pre_steps', 'recon', 'recon_loss_mask', 'recon_reset_mask', 'recon_vel', 'reset_mask', 'success', 't_gt', 't_recon', 'v_gt_speed', 'v_gt_u', 'v_gt_v', 'v_gt_vec', 'v_recon_speed', 'v_recon_u', 'v_recon_v', 'v_recon_vec', 'vel_mask', 'vs_gt_speed', 'vs_gt_u', 'vs_gt_v', 'vs_gt_vec', 'vs_recon_speed', 'vs_recon_u', 'vs_recon_v', 'vs_recon_vec', 'window_x_max', 'window_x_min', 'window_y_max', 'window_y_min']
gt: shape=(11, 8, 2), dtype=float64
recon: shape=(11, 8, 2), dtype=float64
loss_mask: shape=(11, 8), dtype=float64
reset_mask: shape=(11, 8), dtype=float64
v_gt_speed: shape=(), dtype=object, val=None
v_recon_speed: shape=(), dtype=object, val=None
RAD: shape=(), dtype=float64, val=0.002
cyl_center_x: shape=(), dtype=float64, val=0.0075
cyl_center_y: shape=(), dtype=float64, v

In [ ]:
# =================================================================================
# Velocity fields visualization (Task05 Experimental Cylinder, inversion-range view)
## Row 1: vn-GT, vs-GT
## Row 2: vn-Recon, vs-Recon 
# =================================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, patches
from IPython.display import HTML
import matplotlib as mpl
from matplotlib.colors import ListedColormap

bmap = mpl.cm.twilight_shifted
rmap = mpl.cm.twilight
bluecolors = bmap(np.linspace(0, 1, 256))
redcolors  = rmap(np.linspace(0, 1, 256))
cblue = ListedColormap(bluecolors[:100])
cred  = ListedColormap(redcolors[128:228])

# ---------- load data ----------
result_path = '../data/experiment/cyl_exp_recon.npz'

with np.load(result_path, allow_pickle=True) as data:
    Lx = float(data['Lx']) if 'Lx' in data else 0.0206
    Ly = float(data['Ly']) if 'Ly' in data else 0.2
    Wx = float(data['Wx']) if 'Wx' in data else Lx
    Wy = float(data['Wy']) if 'Wy' in data else 0.0227
    RAD = float(data['RAD']) if 'RAD' in data else 0.001
    cyl_cx = float(data['cyl_center_x']) if 'cyl_center_x' in data else Lx / 2
    cyl_cy = float(data['cyl_center_y']) if 'cyl_center_y' in data else Ly/2-Wy/2 + Wy * 0.56

    saved_window_x_min = float(data['window_x_min']) if 'window_x_min' in data else 0.0
    saved_window_x_max = float(data['window_x_max']) if 'window_x_max' in data else Lx
    saved_window_y_min = float(data['window_y_min']) if 'window_y_min' in data else (Ly/2-Wy/2)
    saved_window_y_max = float(data['window_y_max']) if 'window_y_max' in data else (Ly/2+Wy/2)

    v_gt_u = data.get('v_gt_u')
    v_gt_v = data.get('v_gt_v')
    v_gt_speed = data.get('v_gt_speed')
    v_recon_u = data.get('v_recon_u')
    v_recon_v = data.get('v_recon_v')
    v_recon_speed = data.get('v_recon_speed')

    vs_gt_u = data.get('vs_gt_u')
    vs_gt_v = data.get('vs_gt_v')
    vs_gt_speed = data.get('vs_gt_speed')
    vs_recon_u = data.get('vs_recon_u')
    vs_recon_v = data.get('vs_recon_v')
    vs_recon_speed = data.get('vs_recon_speed')

    gt = data.get('gt')
    recon = data.get('recon')
    loss_mask = data.get('loss_mask')

    # --- Load Supervision and Sim pts + velocities ---
    supervision_pts = data.get('supervision_pts')
    supervision_pts_vel = data.get('supervision_pts_vel')
    supervision_pts_mask = data.get('supervision_pts_mask')
    
    sim_pts = data.get('sim_pts')
    sim_pts_vel = data.get('sim_pts_vel')

    supervision_times = data.get('supervision_times')
    fit_times = data.get('fit_times')
    saved_time_axis = data.get('time_axis') if 'time_axis' in data else None
    saved_times = data.get('times') if 'times' in data else None

# ---------- helpers ----------
def as_time_array(x):
    if x is None:
        return None
    try:
        a = np.array(x)
    except Exception:
        try:
            a = np.array(list(x))
        except Exception:
            return None
    if a.dtype == object:
        try:
            a = np.stack([np.asarray(e, dtype=float) for e in a])
        except Exception:
            return None
    try:
        return a.astype(float)
    except Exception:
        return None

def normalize_marker_series(x):
    a = as_time_array(x)
    if a is None:
        return None
    if a.ndim == 3 and a.shape[-1] == 2:
        return a
    if a.ndim == 3 and a.shape[1] == 2:
        return np.transpose(a, (0, 2, 1))
    if a.ndim == 2 and a.shape[-1] == 2:
        return a[None, ...]
    return None

def normalize_mask_series(x, target_time=None, target_markers=None):
    a = as_time_array(x)
    if a is None:
        return None
    if a.ndim == 2:
        pass
    elif a.ndim == 1:
        a = a[None, ...]
    else:
        return None
    if target_time is not None and a.shape[0] != target_time:
        if a.shape[1] == target_time and (target_markers is None or a.shape[0] == target_markers):
            a = a.T
    return a.astype(float)

def resolve_saved_time_axis(*candidates):
    for cand in candidates:
        arr = as_time_array(cand)
        if arr is None:
            continue
        arr = np.asarray(arr, dtype=float).reshape(-1)
        arr = arr[np.isfinite(arr)]
        if arr.size >= 2:
            return arr
    return None

v_gt_speed = as_time_array(v_gt_speed)
v_recon_speed = as_time_array(v_recon_speed)
v_gt_u = as_time_array(v_gt_u)
v_gt_v = as_time_array(v_gt_v)
v_recon_u = as_time_array(v_recon_u)
v_recon_v = as_time_array(v_recon_v)
vs_gt_speed = as_time_array(vs_gt_speed)
vs_recon_speed = as_time_array(vs_recon_speed)
vs_gt_u = as_time_array(vs_gt_u)
vs_gt_v = as_time_array(vs_gt_v)
vs_recon_u = as_time_array(vs_recon_u)
vs_recon_v = as_time_array(vs_recon_v)
gt = normalize_marker_series(gt)
recon = normalize_marker_series(recon)

# ================== Restrict T and time_axis to supervision_pts timeline ==================
if supervision_pts is not None and hasattr(supervision_pts, 'shape') and supervision_pts.shape[0] > 0:
    T = supervision_pts.shape[0]
    print(f"[INFO] Using supervision_pts timeline: T={T}")
    # Try to get the correct time axis for supervision_pts
    time_axis_saved = resolve_saved_time_axis(supervision_times, saved_time_axis, saved_times, fit_times)
    if time_axis_saved is not None and len(time_axis_saved) >= T:
        time_axis = time_axis_saved[:T]
    else:
        time_axis = np.arange(T, dtype=float)
else:
    # fallback to previous logic
    T_candidates = []
    for arr in (v_gt_speed, v_recon_speed, v_gt_u, v_recon_u, vs_gt_speed, vs_recon_speed, gt, recon):
        if arr is not None:
            try:
                T_candidates.append(int(np.asarray(arr).shape[0]))
            except Exception:
                pass
    T = max(T_candidates) if T_candidates else 1
    time_axis_saved = resolve_saved_time_axis(supervision_times, saved_time_axis, saved_times, fit_times)
    if time_axis_saved is not None:
        if len(time_axis_saved) != T:
            T_old = T
            T = min(T, len(time_axis_saved))
            print(f'Aligning frame count to saved timeline: data_frames={T_old}, saved_times={len(time_axis_saved)}, used={T}')
        time_axis = time_axis_saved[:T]
    else:
        time_axis = np.arange(T, dtype=float)

marker_count = None
for arr in (gt, recon):
    if arr is not None and arr.ndim == 3:
        marker_count = arr.shape[1]
        break
loss_mask = normalize_mask_series(loss_mask, target_time=T, target_markers=marker_count)

# ---------- local window ----------
y_min = saved_window_y_min
y_max = saved_window_y_max
x_min = saved_window_x_min
x_max = saved_window_x_max
obs_center = (cyl_cx, cyl_cy)
subplot_aspect = 'equal'

print(f'View window: x=[{x_min:.5f}, {x_max:.5f}]')
print(f'             y=[{y_min:.5f}, {y_max:.5f}]')
print(f'Cylinder centre: ({cyl_cx:.5f}, {cyl_cy:.5f})  R={RAD}')
print(f'Visualization timeline frames: {len(time_axis)} (from saved time arrays)')

# ---------- meshgrid for streamplot ----------
X = Y = None
u_shape = None
for arr in (v_gt_u, v_recon_u, vs_gt_u, vs_recon_u):
    if arr is not None:
        u_shape = arr.shape
        break

if u_shape is not None and len(u_shape) >= 3:
    _, d1, d2 = u_shape
    if d1 > d2:
        H, W = d1, d2
        is_transposed = False
    else:
        H, W = d2, d1
        is_transposed = True
    x_vec = np.linspace(0, Lx, W)
    y_vec = np.linspace(0, Ly, H)
    X, Y = np.meshgrid(x_vec, y_vec)
    y_idx = np.where((y_vec >= y_min) & (y_vec <= y_max))[0]
else:
    H, W = 120, 120
    is_transposed = False
    y_idx = None

def get_local_min_max(speed_field, y_indices):
    if speed_field is None:
        return 0.0, 1.0
    if y_indices is None or len(y_indices) == 0:
        return float(np.nanmin(speed_field)), float(np.nanmax(speed_field))
    if not is_transposed:
        local = speed_field[:, y_indices, :]
    else:
        local = speed_field[:, :, y_indices]
    return float(np.nanmin(local)), float(np.nanmax(local))

vmin_gt, vmax_gt = get_local_min_max(v_gt_speed, y_idx)
vmin_re, vmax_re = get_local_min_max(v_recon_speed, y_idx)
vs_vmin_gt, vs_vmax_gt = get_local_min_max(vs_gt_speed, y_idx)
vs_vmin_re, vs_vmax_re = get_local_min_max(vs_recon_speed, y_idx)

# --- Helper to extract and mask both points and velocities ---
def get_pts_and_vel(pts, vel, mask, i):
    if pts is None or pts.ndim != 3 or i >= pts.shape[0]:
        return None, None
    p = np.asarray(pts[i], dtype=float)
    if p.ndim != 2 or p.shape[1] != 2:
        return None, None
    
    v = None
    if vel is not None and i < vel.shape[0]:
        v = np.asarray(vel[i], dtype=float)
    
    if mask is not None and i < mask.shape[0]:
        try:
            visible = np.asarray(mask[i]).reshape(-1) > 0.5
            if visible.shape[0] == p.shape[0]:
                p = p[visible]
                if v is not None:
                    v = v[visible]
        except Exception:
            pass
    return (p if p.size > 0 else None), (v if (v is not None and v.size > 0) else None)

def clip_pts_and_vel_to_window(points, velocities, x_min_val, x_max_val, y_min_val, y_max_val):
    if points is None or points.size == 0:
        return None, None, 0
    vis = ((points[:, 0] >= x_min_val) & (points[:, 0] <= x_max_val) &
           (points[:, 1] >= y_min_val) & (points[:, 1] <= y_max_val))
    clipped_p = points[vis]
    clipped_v = velocities[vis] if velocities is not None else None
    return (clipped_p if clipped_p.size else None), (clipped_v if (clipped_v is not None and clipped_v.size) else None), int(np.sum(vis))


fig, axs = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
titles = ['vn GT', 'vs GT', 'vn Recon', 'vs Recon']

init_vn = np.zeros((H, W))
init_vs = np.zeros((H, W))
im0 = axs[0, 0].imshow(init_vn, origin='lower', extent=[0, Lx, 0, Ly], cmap=cred, vmin=vmin_gt, vmax=vmax_gt, interpolation='bilinear')
im1 = axs[0, 1].imshow(init_vs, origin='lower', extent=[0, Lx, 0, Ly], cmap=cblue, vmin=vs_vmin_gt, vmax=vs_vmax_gt, interpolation='bilinear')
im2 = axs[1, 0].imshow(init_vn, origin='lower', extent=[0, Lx, 0, Ly], cmap=cred, vmin=vmin_re, vmax=vmax_re, interpolation='bilinear')
im3 = axs[1, 1].imshow(init_vs, origin='lower', extent=[0, Lx, 0, Ly], cmap=cblue, vmin=vs_vmin_re, vmax=vs_vmax_re, interpolation='bilinear')

for ax in axs.ravel():
    ax.add_patch(patches.Circle(obs_center, RAD, facecolor='gray', edgecolor='black', zorder=15))
    ax.set_xlim(0, Lx)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect(subplot_aspect)
    ax.set_ylabel('y [m]')
    ax.set_xlabel('x [m]')
    ax.axhline(saved_window_y_min, color='cyan', ls='--', lw=0.8, alpha=0.7, zorder=9)
    ax.axhline(saved_window_y_max, color='cyan', ls='--', lw=0.8, alpha=0.7, zorder=9)

fig.colorbar(im0, ax=axs[0, 0], fraction=0.046, pad=0.04).set_label('vn speed (local)')
fig.colorbar(im1, ax=axs[0, 1], fraction=0.046, pad=0.04).set_label('vs speed (local)')
fig.colorbar(im2, ax=axs[1, 0], fraction=0.046, pad=0.04).set_label('vn speed (local)')
fig.colorbar(im3, ax=axs[1, 1], fraction=0.046, pad=0.04).set_label('vs speed (local)')

s0, s1, s2, s3 = [None], [None], [None], [None]

# Placeholders for Scatters and Quivers (Arrows)
scatter_gt = [None, None]
quiver_gt = [None, None]
scatter_re = [None, None]
quiver_re = [None, None]

def _remove_streamplot(sp):
    if sp is None:
        return
    try:
        sp.lines.remove()
    except Exception:
        pass
    try:
        for a in sp.arrows:
            a.remove()
    except Exception:
        pass


def update(i):
    if v_gt_u is not None and v_gt_v is not None and i < v_gt_u.shape[0]:
        u, v = np.asarray(v_gt_u[i], dtype=float), np.asarray(v_gt_v[i], dtype=float)
        if is_transposed:
            u, v = u.T, v.T
        im0.set_data(np.sqrt(u ** 2 + v ** 2))
        _remove_streamplot(s0[0])
        s0[0] = axs[0, 0].streamplot(X, Y, u, v, color='whitesmoke', density=1.5, linewidth=0.7) if X is not None else None

    if vs_gt_u is not None and vs_gt_v is not None and i < vs_gt_u.shape[0]:
        u2, v2 = np.asarray(vs_gt_u[i], dtype=float), np.asarray(vs_gt_v[i], dtype=float)
        if is_transposed:
            u2, v2 = u2.T, v2.T
        im1.set_data(np.sqrt(u2 ** 2 + v2 ** 2))
        _remove_streamplot(s1[0])
        s1[0] = axs[0, 1].streamplot(X, Y, u2, v2, color='whitesmoke', density=1.5, linewidth=0.7) if X is not None else None

    if v_recon_u is not None and v_recon_v is not None and i < v_recon_u.shape[0]:
        ur, vr = np.asarray(v_recon_u[i], dtype=float), np.asarray(v_recon_v[i], dtype=float)
        if is_transposed:
            ur, vr = ur.T, vr.T
        im2.set_data(np.sqrt(ur ** 2 + vr ** 2))
        _remove_streamplot(s2[0])
        s2[0] = axs[1, 0].streamplot(X, Y, ur, vr, color='whitesmoke', density=5.5, linewidth=0.7) if X is not None else None

    if vs_recon_u is not None and vs_recon_v is not None and i < vs_recon_u.shape[0]:
        u2r, v2r = np.asarray(vs_recon_u[i], dtype=float), np.asarray(vs_recon_v[i], dtype=float)
        if is_transposed:
            u2r, v2r = u2r.T, v2r.T
        im3.set_data(np.sqrt(u2r ** 2 + v2r ** 2))
        _remove_streamplot(s3[0])
        s3[0] = axs[1, 1].streamplot(X, Y, u2r, v2r, color='whitesmoke', density=5.5, linewidth=0.7) if X is not None else None

    # --- Extract Supervision and Sim Pts with Velocities ---
    sup_p, sup_v = get_pts_and_vel(supervision_pts, supervision_pts_vel, supervision_pts_mask, i)
    sim_p, sim_v = get_pts_and_vel(sim_pts, sim_pts_vel, None, i)

    # Clip pts and vel to visualization window limits (using 0 to Lx)
    sup_p, sup_v, n_sup = clip_pts_and_vel_to_window(sup_p, sup_v, 0, Lx, y_min, y_max)
    sim_p, sim_v, n_sim = clip_pts_and_vel_to_window(sim_p, sim_v, 0, Lx, y_min, y_max)

    # Remove previous scatter and quiver plots to update the frame
    for k in range(2):
        if scatter_gt[k] is not None: scatter_gt[k].remove()
        if quiver_gt[k] is not None: quiver_gt[k].remove()
        if scatter_re[k] is not None: scatter_re[k].remove()
        if quiver_re[k] is not None: quiver_re[k].remove()

        scatter_gt[k] = None
        quiver_gt[k] = None
        scatter_re[k] = None
        quiver_re[k] = None

    # --- Plot Supervision Pts (GT Subplots: Row 1) ---
    if sup_p is not None:
        for k in range(2):
            # Scatter for Positions
            scatter_gt[k] = axs[0, k].scatter(sup_p[:, 0], sup_p[:, 1], s=0, facecolors='lightblue', edgecolors='white', linewidths=0.6, zorder=11)
            # Quiver for Velocities
            if sup_v is not None:
                quiver_gt[k] = axs[0, k].quiver(sup_p[:, 0], sup_p[:, 1], sup_v[:, 0], sup_v[:, 1], color='yellow', width=0.006, zorder=12)

    # --- Plot Sim Pts (Recon Subplots: Row 2) ---
    if sim_p is not None:
        for k in range(2):
            # Scatter for Positions
            scatter_re[k] = axs[1, k].scatter(sim_p[:, 0], sim_p[:, 1], s=0, facecolors='lightgreen', edgecolors='black', linewidths=0.6, zorder=11)
            # Quiver for Velocities
            if sim_v is not None:
                quiver_re[k] = axs[1, k].quiver(sim_p[:, 0], sim_p[:, 1], sim_v[:, 0], sim_v[:, 1], color='lightgreen', edgecolors='black', linewidths=0.1, width=0.006, zorder=12)

    time_label = f'{time_axis[i]:.6f}s' if i < len(time_axis) else str(i)
    axs[0, 0].set_title(f'{titles[0]}  t={time_label}  (sup_vis={n_sup})')
    axs[0, 1].set_title(f'{titles[1]}  t={time_label}')
    axs[1, 0].set_title(f'{titles[2]}  t={time_label}  (sim_vis={n_sim})')
    axs[1, 1].set_title(f'{titles[3]}  t={time_label}')
    return im0, im1, im2, im3

num_frames = min(20, T)
anim = animation.FuncAnimation(fig, update, frames=num_frames, interval=300, blit=False)
plt.close(fig)
gif_path = '../data/experiment/cyl_exp_recon.gif'
anim.save(gif_path)
print(f'Saved GIF to {gif_path}')
# HTML(f"<b>Saved visualization to:</b> {gif_path}")

MovieWriter ffmpeg unavailable; using Pillow instead.


View window: x=[0.00000, 0.01500]
             y=[0.10450, 0.12150]
Cylinder centre: (0.00750, 0.11300)  R=0.002
Continuous-fit mode detected with 11 fitted times in [0, 0.100] s
Saved GIF to ../data/experiment/cyl_exp_recon.gif
Saved GIF to ../data/experiment/cyl_exp_recon.gif


In [ ]:
# === Visualize supervision points (used as pts for inversion) ===

with np.load(result_path, allow_pickle=True) as data:
    supervision_pts = data.get('supervision_pts')
    supervision_pts_mask = data.get('supervision_pts_mask')

# Helper to select and mask supervision points
def select_supervision_points(pts, mask, i):
    if pts is None or pts.ndim != 3 or i >= pts.shape[0]:
        return None
    p = np.asarray(pts[i], dtype=float)
    if p.ndim != 2 or p.shape[1] != 2:
        return None
    if mask is not None and i < mask.shape[0]:
        try:
            visible = np.asarray(mask[i]).reshape(-1) > 0.5
            if visible.shape[0] == p.shape[0]:
                p = p[visible]
        except Exception:
            pass
    return p if p.size > 0 else None

# Add to animation: plot supervision_pts as 'pts' in recon subplot
def update_with_supervision(i):
    # ...existing update logic...
    # Add supervision points to recon subplot (bottom left)
    sup_pts = select_supervision_points(supervision_pts, supervision_pts_mask, i)
    if sup_pts is not None:
        axs[1, 0].scatter(sup_pts[:, 0], sup_pts[:, 1], s=30, facecolors='orange', edgecolors='black', linewidths=0.6, zorder=12, label='Supervision pts')
    # ...existing update logic...

# To use: replace the call to 'update' in FuncAnimation with 'update_with_supervision' if you want to show supervision pts as 'pts'

# 检查 supervision_pts 粒子是否在可视化窗口内，并统计每帧数量

def count_pts_in_window(pts, mask, x_min, x_max, y_min, y_max):
    counts = []
    if pts is None or pts.ndim != 3:
        print('No supervision_pts loaded or wrong shape')
        return counts
    n_frames = pts.shape[0]
    for i in range(n_frames):
        p = np.asarray(pts[i], dtype=float)
        if p.ndim != 2 or p.shape[1] != 2:
            counts.append(0)
            continue
        if mask is not None and i < mask.shape[0]:
            try:
                visible = np.asarray(mask[i]).reshape(-1) > 0.5
                if visible.shape[0] == p.shape[0]:
                    p = p[visible]
            except Exception:
                pass
        in_window = (p[:, 0] >= x_min) & (p[:, 0] <= x_max) & (p[:, 1] >= y_min) & (p[:, 1] <= y_max)
        n_in = int(np.sum(in_window))
        counts.append(n_in)
    return counts

# 使用保存的窗口参数
with np.load(result_path, allow_pickle=True) as data:
    x_min = float(data['window_x_min']) if 'window_x_min' in data else 0.0
    x_max = float(data['window_x_max']) if 'window_x_max' in data else 0.015
    y_min = float(data['window_y_min']) if 'window_y_min' in data else 0.0
    y_max = float(data['window_y_max']) if 'window_y_max' in data else 0.2

counts = count_pts_in_window(supervision_pts, supervision_pts_mask, x_min, x_max, y_min, y_max)
for i, n in enumerate(counts):
    print(f'Frame {i}: {n} supervision pts in window')
if counts:
    print(f'Max: {max(counts)}, Min: {min(counts)}, Mean: {np.mean(counts):.2f} pts per frame in window')